# Collaborative Filtering

Memory-Based Algorithm
- Item based
- **User based: 우선 dot product 없는 버전 구현 해볼 것 (Item based와 방식 동일)**


Model-Based Algorithm
- Latent Factor 협업 필터링 방법 (Matrix Factorization)

# 구글 드라이브 연결

파일 > 드라이브 마운트 클릭

In [ ]:
import os
from pathlib import Path
DATA_DIR = Path("drive/MyDrive/kw/추천특강/강의자료/data/")
os.listdir(DATA_DIR)

# 데이터 로드


In [ ]:
MOVIE_DATA_PATH = f"{DATA_DIR}/movies_refined.csv"
RATING_DATA_PATH = f"{DATA_DIR}/ratings_refined.csv"

로드 후 데이터 타입 형변환  
id는 문자열로 처리

영화 데이터
```
movie_id: str
```

평점 데이터
```
user_id: str
movie_id: str
```

In [ ]:
import pandas as pd  ##pandas를 사용할시에는 아래처럼시작을 해줘야함. 그냥필수임.

movies_df = pd.read_csv(MOVIE_DATA_PATH,
    usecols=['movie_id', 'title'],
    dtype={'movie_id': str})

ratings_df = pd.read_csv(RATING_DATA_PATH,
    usecols=['user_id', 'movie_id', 'rating'],
    dtype={'user_id': str, 'movie_id': str})

In [ ]:
movie_ratings_df = pd.merge(ratings_df, movies_df, on='movie_id', how='left')
movie_ratings_df

,user_id,movie_id,rating,title
0,1,1,4.0,Toy Story (1995)
1,1,3,4.0,Grumpier Old Men (1995)
2,1,6,4.0,Heat (1995)
3,1,47,5.0,Seven (a.k.a. Se7en) (1995)
4,1,50,5.0,"Usual Suspects, The (1995)"
...,...,...,...,...
100784,610,166534,4.0,Split (2017)
100785,610,168248,5.0,John Wick: Chapter Two (2017)
100786,610,168250,5.0,Get Out (2017)
100787,610,168252,5.0,Logan (2017)


null 값 체크

In [ ]:
movie_ratings_df.columns[movie_ratings_df.isna().any()].tolist()

[]

영화명 결측치 체크

In [ ]:
movie_ratings_df[movie_ratings_df['title'].isnull()]

,user_id,movie_id,rating,title


# 유저 유사도 행렬 준비


유저X 입력되었을 때

유저X와 비슷한 유저 찾는데 사용할 행렬

In [ ]:
user_movie_df = movie_ratings_df.pivot_table(values='rating', index='user_id', columns='title')
user_movie_df

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
user_id,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN
10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
101,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN
102,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2.5,3.0,NaN,NaN,NaN
96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
97,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# rating 정보 있는 영화 개수
movie_ratings_df['title'].nunique()

9685

In [ ]:
# 610 x 9685 행렬
# 사용자를 9685차원의 벡터로 보려는 것
user_movie_df.shape

(610, 9685)

## 결측치 처리

null값이 있으면 cosine similarity 함수가 안돌아감

하지만, null값을 0으로 치환하고 계산할경우 결과가 달라짐

(마치 해당 영화를 보고 0점을 준것으로 계산)

In [ ]:
tmp_user_movie_df = user_movie_df.copy().fillna(0)

## 유사도 행렬 계산

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity_matrix = cosine_similarity(tmp_user_movie_df)
user_similarity_matrix.shape

(610, 610)

In [ ]:
user_similarity_matrix

array([[1.        , 0.01687529, 0.14776299, ..., 0.08655715, 0.06218738,
        0.06623056],
       [0.01687529, 1.        , 0.09047407, ..., 0.05777051, 0.13022326,
        0.02058682],
       [0.14776299, 0.09047407, 1.        , ..., 0.07811067, 0.05303927,
        0.0790632 ],
       ...,
       [0.08655715, 0.05777051, 0.07811067, ..., 1.        , 0.08051508,
        0.        ],
       [0.06218738, 0.13022326, 0.05303927, ..., 0.08051508, 1.        ,
        0.        ],
       [0.06623056, 0.02058682, 0.0790632 , ..., 0.        , 0.        ,
        1.        ]])

## 데이터 프레임화

In [ ]:
user_ids = user_movie_df.index
user_ids

Index(['1', '10', '100', '101', '102', '103', '104', '105', '106', '107',
       ...
       '90', '91', '92', '93', '94', '95', '96', '97', '98', '99'],
      dtype='object', name='user_id', length=610)

In [ ]:
# 유저와 유저 간에 대한 유사도  #협업의 용어. 다시확인 11:09
user_similarity_df = pd.DataFrame(user_similarity_matrix,
                                index=user_ids, columns=user_ids)
print(user_similarity_df.shape)
user_similarity_df.head()

(610, 610)


user_id,1,10,100,101,102,103,104,105,106,107,...,90,91,92,93,94,95,96,97,98,99
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.016875,0.147763,0.141447,0.156466,0.211480,0.121173,0.090396,0.019425,0.066894,...,0.051841,0.334727,0.028767,0.232766,0.110082,0.178069,0.282780,0.086557,0.062187,0.066231
10,0.016875,1.000000,0.090474,0.020533,0.036215,0.116766,0.178308,0.149666,0.133146,0.003137,...,0.000000,0.055251,0.016773,0.029036,0.038062,0.042865,0.018888,0.057771,0.130223,0.020587
100,0.147763,0.090474,1.000000,0.070894,0.142033,0.136122,0.178267,0.065512,0.000000,0.098179,...,0.015134,0.189074,0.000000,0.122328,0.124926,0.093664,0.071043,0.078111,0.053039,0.079063
101,0.141447,0.020533,0.070894,1.000000,0.026755,0.105728,0.045393,0.065482,0.000000,0.000000,...,0.000000,0.133919,0.000000,0.012955,0.000000,0.070774,0.107449,0.103712,0.000000,0.000000
102,0.156466,0.036215,0.142033,0.026755,1.000000,0.094133,0.087527,0.051604,0.029935,0.353838,...,0.000000,0.259438,0.000000,0.219928,0.586507,0.046217,0.136751,0.048376,0.054637,0.428520


## 유사도 행렬 저장

인덱스가 1부터 시작하는 user_id 정보이므로

저장할 때 `index=True` 옵션 필요

In [ ]:
# 파일명에도 인덱스 포함 여부 명시하면 명확성 증가
user_similarity_file_path = DATA_DIR / "user_similarity_with_index.csv"
user_similarity_file_path

PosixPath('drive/MyDrive/kw/추천특강/강의자료/data/user_similarity_with_index.csv')

In [ ]:
# user_id로 사용하는 인덱스도 저장 필요한 정보  #애초에 저장할때 이렇게 저장해야합니다.
user_similarity_df.to_csv(user_similarity_file_path, index=True)
user_similarity_df

user_id,1,10,100,101,102,103,104,105,106,107,...,90,91,92,93,94,95,96,97,98,99
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.016875,0.147763,0.141447,0.156466,0.211480,0.121173,0.090396,0.019425,0.066894,...,0.051841,0.334727,0.028767,0.232766,0.110082,0.178069,0.282780,0.086557,0.062187,0.066231
10,0.016875,1.000000,0.090474,0.020533,0.036215,0.116766,0.178308,0.149666,0.133146,0.003137,...,0.000000,0.055251,0.016773,0.029036,0.038062,0.042865,0.018888,0.057771,0.130223,0.020587
100,0.147763,0.090474,1.000000,0.070894,0.142033,0.136122,0.178267,0.065512,0.000000,0.098179,...,0.015134,0.189074,0.000000,0.122328,0.124926,0.093664,0.071043,0.078111,0.053039,0.079063
101,0.141447,0.020533,0.070894,1.000000,0.026755,0.105728,0.045393,0.065482,0.000000,0.000000,...,0.000000,0.133919,0.000000,0.012955,0.000000,0.070774,0.107449,0.103712,0.000000,0.000000
102,0.156466,0.036215,0.142033,0.026755,1.000000,0.094133,0.087527,0.051604,0.029935,0.353838,...,0.000000,0.259438,0.000000,0.219928,0.586507,0.046217,0.136751,0.048376,0.054637,0.428520
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.178069,0.042865,0.093664,0.070774,0.046217,0.184223,0.125659,0.143146,0.044271,0.000000,...,0.061351,0.265121,0.023369,0.091787,0.054546,1.000000,0.208704,0.089793,0.054944,0.023075
96,0.282780,0.018888,0.071043,0.107449,0.136751,0.177743,0.087595,0.083982,0.021467,0.130466,...,0.044602,0.271696,0.000000,0.240489,0.204494,0.208704,1.000000,0.144190,0.057705,0.072585
97,0.086557,0.057771,0.078111,0.103712,0.048376,0.102086,0.048995,0.141523,0.075348,0.000000,...,0.000000,0.150644,0.000000,0.045750,0.044044,0.089793,0.144190,1.000000,0.080515,0.000000


# User-based CF

사용자 유사도 기반 추천

유저X 입력되었을 때

유저X와 비슷한 유저 검색 -> Y, Z

유저 Y, Z가 많이 본 영화 추천

In [ ]:
# 샘플 사용자 (ID 문자열 타입으로 사용중)
user_id = "1"

In [ ]:
# 유사한 사용자 10명
user_similarity_df[user_id].sort_values(ascending=False)[1:11]

,1
user_id,
266,0.357408
313,0.351562
368,0.345127
57,0.345034
91,0.334727
469,0.330664
39,0.329782
288,0.329700
452,0.328048


In [ ]:
similar_users_df = user_similarity_df[user_id].sort_values(ascending=False)[1:11]
similar_users_df = similar_users_df.reset_index()
similar_users_df

,user_id,1
0,266,0.357408
1,313,0.351562
2,368,0.345127
3,57,0.345034
4,91,0.334727
5,469,0.330664
6,39,0.329782
7,288,0.329700
8,452,0.328048
9,45,0.327922


In [ ]:
tmp_similar_user_id = similar_users_df.loc[0, 'user_id']
tmp_similar_user_id

'266'

In [ ]:
tmp_movie_title = (
    movie_ratings_df
    .loc[movie_ratings_df['user_id'] == tmp_similar_user_id, 'title']
    .tolist()[0]
)
tmp_movie_title

'Toy Story (1995)'

In [ ]:
titles = []
for uid in similar_users_df['user_id'].tolist():
    title = (
        movie_ratings_df
        .loc[movie_ratings_df['user_id'] == uid, 'title']
        .tolist()[0]
    )
    titles.append(title)
list(set(titles))

['Heat (1995)',
 'Twelve Monkeys (a.k.a. 12 Monkeys) (1995)',
 'GoldenEye (1995)',
 'Grumpier Old Men (1995)',
 'Toy Story (1995)']

In [ ]:
# 사용자와 유사한 사람들이 많이 본 영화 추천
def get_recomendation(
        movie_ratings_df,
        user_similarity_df,
        user_id
    ):
    similar_users_df = user_similarity_df[user_id].sort_values(ascending=False)[1:11]
    similar_users_df = similar_users_df.reset_index()

    titles = []
    for uid in similar_users_df['user_id'].tolist():
        title = (
            movie_ratings_df
            .loc[movie_ratings_df['user_id'] == uid, 'title']
            .tolist()[0]
        )
        titles.append(title)

    return list(set(titles))

In [ ]:
get_recomendation(movie_ratings_df, user_similarity_df, user_id)

['Heat (1995)',
 'Twelve Monkeys (a.k.a. 12 Monkeys) (1995)',
 'GoldenEye (1995)',
 'Grumpier Old Men (1995)',
 'Toy Story (1995)']